# 🧹 BƯỚC 2: Tiền xử lý dữ liệu (Preprocessing)
**Amazon Clothing Review Analysis & Recommendation**

---
### 🎯 Mục tiêu của Bước 2:
1. Load toàn bộ dữ liệu với **sampling thông minh** (5% ≈ ~1.1M reviews)
2. Làm sạch dữ liệu (xử lý missing, duplicate, outlier)
3. Chuẩn hóa kiểu dữ liệu
4. Tạo nhãn **Sentiment** từ rating
5. Merge review + meta
6. Lưu kết quả dạng `.parquet` để dùng cho các bước sau

### 💡 Tại sao dùng Sampling?
- File gốc: **22.6M reviews** (~5GB) → không thể load hết vào RAM
- Sample **5%** → ~**1.1M reviews** → đủ lớn để mô hình học tốt, đủ nhỏ để xử lý nhanh
- Sampling **ngẫu nhiên đều** → giữ được tính đại diện của dữ liệu

## Cell 2.1 — Import & Setup

In [1]:
# ============================================================
# CELL 2.1: Import thư viện & cấu hình đường dẫn
# ============================================================

import pandas as pd
import numpy as np
import json, gzip, os, sys, re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm         # Thanh tiến trình khi load data lớn

# Đường dẫn
ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"

# Tạo thư mục nếu chưa có
for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# SAMPLING CONFIG
# Thay đổi các tham số này nếu cần
# ============================================================
SAMPLE_RATE           = 0.05    # 5% review  (~1.1M dòng)
MAX_META_ROWS         = 500_000 # 500k sản phẩm
MIN_TEXT_LENGTH       = 10      # Review phải có ít nhất 10 ký tự
MIN_REVIEWS_PER_USER  = 5       # User phải có ít nhất 5 reviews (cho Recommendation)
MIN_REVIEWS_PER_ITEM  = 5       # Item phải có ít nhất 5 reviews
RANDOM_SEED           = 42

np.random.seed(RANDOM_SEED)
print("✅ Setup hoàn tất!")
print(f"   Sample rate  : {SAMPLE_RATE*100:.0f}% reviews")
print(f"   Max meta rows: {MAX_META_ROWS:,}")

✅ Setup hoàn tất!
   Sample rate  : 5% reviews
   Max meta rows: 500,000


Tiền xử lý review

In [ ]:
# ============================================================
# CELL 2.4: Tiền xử lý Review - từng bước chi tiết
# ============================================================

def preprocess_review(df):
    df = df.copy()
    log = {}  # Ghi lại số dòng sau mỗi bước
    log['1_raw'] = len(df)

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # Chỉ giữ các cột có ý nghĩa cho phân tích
    # ----------------------------------------------------------
    needed = ['rating', 'text', 'title', 'user_id',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Chọn cột: giữ lại {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Xử lý missing values
    # rating và text là BẮT BUỘC → drop nếu thiếu
    # các cột khác → fill giá trị mặc định
    # ----------------------------------------------------------
    df = df.dropna(subset=['rating', 'text'])
    df['helpful_vote']      = df.get('helpful_vote', pd.Series(0)).fillna(0)
    df['verified_purchase'] = df.get('verified_purchase', pd.Series(False)).fillna(False)
    log['2_drop_na'] = len(df)
    print(f"B. Drop NA  : {log['1_raw']:,} → {log['2_drop_na']:,} (-{log['1_raw']-log['2_drop_na']:,})")

    # ----------------------------------------------------------
    # BƯỚC C: Chuẩn hóa kiểu dữ liệu
    # ----------------------------------------------------------
    df['rating']        = pd.to_numeric(df['rating'], errors='coerce')
    df['helpful_vote']  = pd.to_numeric(df['helpful_vote'], errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['rating'])  # drop nếu rating không parse được
    df['rating']        = df['rating'].astype(float)
    log['3_dtype'] = len(df)
    print(f"C. Dtype    : {log['2_drop_na']:,} → {log['3_dtype']:,}")

    # ----------------------------------------------------------
    # BƯỚC D: Chuyển timestamp → datetime
    # timestamp gốc là Unix milliseconds (số ms từ 1970)
    # ----------------------------------------------------------
    if 'timestamp' in df.columns:
        df['date']  = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = df['date'].dt.year
        df['month'] = df['date'].dt.month
        print(f"D. Timestamp: range {df['year'].min():.0f} - {df['year'].max():.0f}")

    # ----------------------------------------------------------
    # BƯỚC E: Tạo nhãn Sentiment từ Rating
    # 4-5 sao → positive | 3 sao → neutral | 1-2 sao → negative
    # Đây là SUPERVISED LABEL cho bài toán Sentiment Analysis
    # ----------------------------------------------------------
    def to_sentiment(r):
        if r >= 4:   return 'positive'
        elif r == 3: return 'neutral'
        else:        return 'negative'

    df['sentiment']       = df['rating'].apply(to_sentiment)
    df['sentiment_score'] = df['rating'].apply(lambda r: 1 if r >= 4 else (0 if r == 3 else -1))
    print(f"E. Sentiment: {df['sentiment'].value_counts().to_dict()}")

    # ----------------------------------------------------------
    # BƯỚC F: Lọc review quá ngắn (noise)
    # Review < 10 ký tự không có giá trị phân tích NLP
    # ----------------------------------------------------------
    df['text_length'] = df['text'].astype(str).str.len()
    before = len(df)
    df = df[df['text_length'] >= MIN_TEXT_LENGTH]
    log['4_short_text'] = len(df)
    print(f"F. Short text: {before:,} → {log['4_short_text']:,} (-{before-log['4_short_text']:,} review quá ngắn)")

    # ----------------------------------------------------------
    # BƯỚC G: Loại bỏ duplicate
    # Cùng 1 user review cùng 1 sản phẩm → chỉ giữ lần đầu
    # ----------------------------------------------------------
    before = len(df)
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')
    log['5_dedup'] = len(df)
    print(f"G. Duplicate: {before:,} → {log['5_dedup']:,} (-{before-log['5_dedup']:,} duplicates)")

    # ----------------------------------------------------------
    # BƯỚC H: Reset index
    # ----------------------------------------------------------
    df = df.reset_index(drop=True)

    print(f"\n✅ Kết quả cuối: {len(df):,} reviews")
    return df, log



Tiền xử lý meta

In [14]:
# ============================================================
# CELL 2.5: Tiền xử lý Meta dataset
# ============================================================

def preprocess_meta(df):
    df = df.copy()
    print(f"Raw shape: {df.shape}")

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # ----------------------------------------------------------
    needed = ['parent_asin', 'title', 'price', 'description',
              'categories', 'average_rating', 'rating_number',
              'store', 'main_category']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Giữ cột: {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Drop sản phẩm không có ID hoặc tên
    # ----------------------------------------------------------
    before = len(df)
    df = df.dropna(subset=['parent_asin', 'title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')
    print(f"B. Drop NA/dup: {before:,} → {len(df):,}")

    # ----------------------------------------------------------
    # BƯỚC C: Xử lý Price
    # Giá có thể là '$29.99' hoặc '29.99' → cần chuẩn hóa
    # Lọc outlier: giá <= 0 hoặc > $10,000 là bất thường
    # ----------------------------------------------------------
    if 'price' in df.columns:
        df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        valid_price = df['price'].between(0.01, 10_000)
        print(f"C. Price: {valid_price.sum():,} sản phẩm có giá hợp lệ / {len(df):,}")
        # Giữ cả null price (không xóa sản phẩm, chỉ để null)

    # ----------------------------------------------------------
    # BƯỚC D: Xử lý Description (list → string)
    # description gốc là list các đoạn văn
    # ----------------------------------------------------------
    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else str(x) if pd.notna(x) else ''
        )
        print(f"D. Description: đã chuyển list → string")

    # ----------------------------------------------------------
    # BƯỚC E: Xử lý Categories
    # categories là list lồng nhau → lấy level 1 làm main_category
    # ----------------------------------------------------------
    if 'categories' in df.columns:
        def extract_category(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_category)
        print(f"E. Categories top 5: {df['main_category'].value_counts().head().to_dict()}")

    df = df.reset_index(drop=True)
    print(f"\n✅ Meta kết quả: {df.shape}")
    return df

2.2: đọc+ tiền xử lý và lưu file review

In [ ]:
# Cấu hình đường dẫn
FINAL_REVIEW_CSV = PROCESSED_DIR / "review_clean_all.csv"
CHUNK_SIZE = 500000 

def process_review_with_sentiment_chunks(filepath, chunk_size=500000):
    print(f"📦 Đang xử lý Review (Sentiment) theo cụm {chunk_size:,}...")
    file_exists = False
    
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        current_chunk = []
        for i, line in enumerate(tqdm(f, desc="Streaming Review")):
            try:
                current_chunk.append(json.loads(line.strip()))
            except: continue

            if len(current_chunk) == chunk_size:
                # Chuyển thành DF và Tiền xử lý (Gọi hàm ở Cell 2.4 của bạn)
                df_temp, _ = preprocess_review(pd.DataFrame(current_chunk))
                
                # Lưu luôn vào file CSV (Append)
                df_temp.to_csv(FINAL_REVIEW_CSV, mode='a', index=False, header=not file_exists)
                
                file_exists = True
                current_chunk = [] # Xóa ngay để giải phóng RAM
                del df_temp

        # Lưu phần còn lại
        if current_chunk:
            df_temp, _ = preprocess_review(pd.DataFrame(current_chunk))
            df_temp.to_csv(FINAL_REVIEW_CSV, mode='a', index=False, header=not file_exists)

    print(f"✅ Đã lưu xong file Review cho Sentiment tại: {FINAL_REVIEW_CSV}")

# Chạy bước 1
process_review_with_sentiment_chunks(REVIEW_PATH, CHUNK_SIZE)

Đọc tiền xử lý + lưu file meta

In [ ]:
# ============================================================
# Cấu hình đường dẫn cho file Meta
# ============================================================
FINAL_META_CSV = PROCESSED_DIR / "meta_clean_all.csv"
CHUNK_SIZE = 500000 

def process_meta_with_cleaning_chunks(filepath, chunk_size=500000):
    print(f"📦 Đang tiền xử lý file Meta theo cụm {chunk_size:,}...")
    file_exists = False
    
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        current_chunk = []
        for i, line in enumerate(tqdm(f, desc="Streaming Meta")):
            try:
                current_chunk.append(json.loads(line.strip()))
            except: 
                continue

            if len(current_chunk) == chunk_size:
                # 1. Chuyển thành DataFrame
                df_raw = pd.DataFrame(current_chunk)
                
                # 2. Gọi hàm tiền xử lý Meta (Hàm preprocess_meta trong Cell 2.5 của bạn)
                # Hàm này sẽ xử lý: Drop NA, chuẩn hóa giá, chuyển đổi Categories...
                df_temp = preprocess_meta(df_raw)
                
                # 3. Lưu nối đuôi vào file CSV
                df_temp.to_csv(FINAL_META_CSV, mode='a', index=False, header=not file_exists)
                
                file_exists = True
                current_chunk = [] # Giải phóng danh sách tạm
                del df_raw, df_temp # Giải phóng RAM

        # Xử lý nốt phần dữ liệu dư cuối cùng
        if current_chunk:
            df_raw = pd.DataFrame(current_chunk)
            df_temp = preprocess_meta(df_raw)
            df_temp.to_csv(FINAL_META_CSV, mode='a', index=False, header=not file_exists)

    print(f"✅ Đã lưu xong file Meta đã làm sạch tại: {FINAL_META_CSV}")

# Thực thi xử lý file Meta
process_meta_with_cleaning_chunks(META_PATH, CHUNK_SIZE)

merge 2 file

In [ ]:
import pandas as pd
from tqdm import tqdm
import os

# Cấu hình đường dẫn các file đã tiền xử lý
# FINAL_REVIEW_CSV: File 21.5M dòng đã sạch & lọc Cold-start
# FINAL_META_CSV: File sản phẩm đã làm sạch
MERGED_OUTPUT_CSV = PROCESSED_DIR / "merged_data_final.csv"
CHUNK_SIZE = 500000

# ============================================================
# BƯỚC 1: Load Meta Data vào bộ nhớ
# Chỉ lấy các cột cần thiết để tiết kiệm RAM
# ============================================================
print("📂 Đang nạp dữ liệu Meta vào bộ nhớ...")
cols_meta = ['parent_asin', 'title', 'price', 'main_category']
df_meta = pd.read_csv(FINAL_META_CSV, usecols=cols_meta)

# Đảm bảo không có ID sản phẩm trùng lặp trong Meta để tránh làm tăng số dòng khi merge
df_meta = df_meta.drop_duplicates(subset=['parent_asin'])
print(f"✅ Đã nạp {len(df_meta):,} sản phẩm.")

# ============================================================
# BƯỚC 2: Merge theo từng cụm Review
# ============================================================
print(f"\n🚀 Đang tiến hành Merge Review với Meta theo từng cụm {CHUNK_SIZE:,}...")

review_reader = pd.read_csv(FINAL_REVIEW_CSV, chunksize=CHUNK_SIZE)
file_exists = False

for chunk in tqdm(review_reader, desc="Merging"):
    # Left join: Giữ tất cả review, khớp thêm thông tin sản phẩm
    merged_chunk = pd.merge(
        chunk, 
        df_meta, 
        on='parent_asin', 
        how='left'
    )
    
    # Đổi tên cột title để phân biệt giữa title của review và title của sản phẩm
    if 'title_y' in merged_chunk.columns:
        merged_chunk = merged_chunk.rename(columns={
            'title_x': 'review_title', 
            'title_y': 'product_title'
        })

    # Lưu nối đuôi (Append) ra file tổng
    merged_chunk.to_csv(
        MERGED_OUTPUT_CSV, 
        mode='a', 
        index=False, 
        header=not file_exists
    )
    
    file_exists = True
    del merged_chunk # Giải phóng RAM ngay lập tức

# Giải phóng bộ nhớ Meta
del df_meta

print(f"\n✨ HOÀN THÀNH!")
print(f"✅ File gộp cuối cùng nằm tại: {MERGED_OUTPUT_CSV}")

B1: Tính toán tần suất

In [ ]:

# Cấu hình
INPUT_CSV = PROCESSED_DIR / "merged_data_final.csv"
OUTPUT_PARQUET = PROCESSED_DIR / "all_for_rec_filtered.parquet"
MIN_REVIEWS_PER_ITEM = 5
MIN_REVIEWS_PER_USER = 5

# ============================================================
# BƯỚC 1: Tính toán tập hợp User/Item hợp lệ (Chỉ load cột ID)
# ============================================================
print("🔍 Bước 1: Đang tính toán tần suất User/Item...")
# Chỉ load 2 cột ID để tiết kiệm RAM tối đa
df_ids = pd.read_csv(INPUT_CSV, usecols=['user_id', 'parent_asin'])

# Lọc lặp cho đến khi ổn định (Cold-start algorithm)
for iteration in range(5):
    n_before = len(df_ids)
    
    # Lọc Item
    item_counts = df_ids['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= MIN_REVIEWS_PER_ITEM].index
    df_ids = df_ids[df_ids['parent_asin'].isin(valid_items)]
    
    # Lọc User
    user_counts = df_ids['user_id'].value_counts()
    valid_users = user_counts[user_counts >= MIN_REVIEWS_PER_USER].index
    df_ids = df_ids[df_ids['user_id'].isin(valid_users)]
    
    n_after = len(df_ids)
    print(f"  Vòng lặp {iteration+1}: {n_before:,} -> {n_after:,}")
    if n_before == n_after: break

# Chuyển thành set để tra cứu (lookup) cực nhanh ở bước sau
valid_user_set = set(df_ids['user_id'].unique())
valid_item_set = set(df_ids['parent_asin'].unique())

# Giải phóng RAM ngay lập tức sau khi lấy được danh sách ID
del df_ids
import gc
gc.collect()

Cold start + lưu parquet

In [ ]:
# Đảm bảo các đường dẫn đã chính xác
OUTPUT_PARQUET = "review_for_rec_filtered.parquet"

print("\n🚀 Bước 2: Đang lọc dữ liệu thực tế và lưu sang Parquet...")

# Khởi tạo reader với ép kiểu string để khớp hoàn toàn với valid_set ở Bước 1
chunk_reader = pd.read_csv(
    INPUT_CSV, 
    chunksize=500000, 
    dtype={'user_id': str, 'parent_asin': str} # Ép kiểu để tránh lệch định dạng
)

writer = None

try:
    for chunk in tqdm(chunk_reader, desc="Đang lọc các dòng"):
        # 1. Lọc dòng hợp lệ dựa trên 2 tập set đã có ở Bước 1
        mask = chunk['user_id'].isin(valid_user_set) & chunk['parent_asin'].isin(valid_item_set)
        filtered_chunk = chunk[mask].copy()
        
        if not filtered_chunk.empty:
            # 2. Chuyển DataFrame thành PyArrow Table
            table = pa.Table.from_pandas(filtered_chunk, preserve_index=False)
            
            # 3. Khởi tạo writer ở cụm đầu tiên có dữ liệu
            if writer is None:
                # Dùng snappy compression để file vừa nhẹ vừa đọc nhanh
                writer = pq.ParquetWriter(OUTPUT_PARQUET, table.schema, compression='snappy')
            
            # 4. Ghi dữ liệu vào file
            writer.write_table(table)
            
            # Giải phóng bộ nhớ cụm vừa xử lý
            del filtered_chunk, table

except Exception as e:
    print(f"\n❌ Có lỗi xảy ra trong quá trình lọc: {e}")

finally:
    # Quan trọng: Đóng file để hoàn tất ghi đĩa
    if writer:
        writer.close()
        print(f"\n✨ HOÀN THÀNH!")
        print(f"✅ File Parquet đã được tạo: {OUTPUT_PARQUET}")
        # Kiểm tra dung lượng file mới
        file_size = os.path.getsize(OUTPUT_PARQUET) / (1024**3)
        print(f"📊 Dung lượng file sau khi lọc & nén: {file_size:.2f} GB")
    else:
        print("\n⚠️ Không thể kết thúc ghi file vì không có dữ liệu khớp hoặc lỗi xảy ra sớm.")

Cold start + Lưu CSV

In [ ]:
import pandas as pd
from tqdm import tqdm
import os

# Cấu hình đường dẫn
# INPUT_CSV = "Clothing_Shoes_and_Jewelry.csv" (File 17GB của bạn)
OUTPUT_CSV_FILTERED = PROCESSED_DIR / "review_for_rec_filtered.csv"

print("\n🚀 Bước 2: Đang lọc dữ liệu thực tế và lưu sang CSV...")

# Khởi tạo reader
chunk_reader = pd.read_csv(
    INPUT_CSV, 
    chunksize=500000, 
    dtype={'user_id': str, 'parent_asin': str}
)

file_exists = False # Biến để kiểm tra xem đã ghi tiêu đề (header) chưa

try:
    for chunk in tqdm(chunk_reader, desc="Đang lọc và ghi CSV"):
        # 1. Lọc dòng hợp lệ (Sử dụng 2 set đã có ở Bước 1)
        mask = chunk['user_id'].isin(valid_user_set) & chunk['parent_asin'].isin(valid_item_set)
        filtered_chunk = chunk[mask]
        
        if not filtered_chunk.empty:
            # 2. Ghi nối đuôi vào file CSV
            # mode='a': append (ghi tiếp)
            # header: chỉ ghi tiêu đề cột ở lần đầu tiên (khi file_exists = False)
            filtered_chunk.to_csv(
                OUTPUT_CSV_FILTERED, 
                mode='a', 
                index=False, 
                header=not file_exists,
                encoding='utf-8'
            )
            
            file_exists = True # Sau lần đầu ghi, các lần sau không ghi header nữa
            
            # Giải phóng bộ nhớ cụm
            del filtered_chunk

    if file_exists:
        print(f"\n✨ HOÀN THÀNH!")
        print(f"✅ File CSV đã được lọc: {OUTPUT_CSV_FILTERED}")
        # Kiểm tra dung lượng
        size_gb = os.path.getsize(OUTPUT_CSV_FILTERED) / (1024**3)
        print(f"📊 Dung lượng file kết quả: {size_gb:.2f} GB")
    else:
        print("\n⚠️ Không tìm thấy dữ liệu nào khớp với bộ lọc.")

except Exception as e:
    print(f"\n❌ Lỗi trong quá trình xử lý: {e}")